# 数据管理与模型训练

本 Demo 分别演示 Episode 转换、数据快照、策略训练、世界模型训练、模型登记与推演 API。


## 0. 初始化平台

只定义工作区与输出目录；数据、训练器、配方和随机种子均在其 API 单元格中绑定。


In [ ]:
# ruff: noqa
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("未找到 PhysReal-X 项目根目录。")
    PROJECT_ROOT = PROJECT_ROOT.parent
for import_root in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))
from examples._common import operation_receipt, write_summary
from physrealx import MethodCapability, PhysRealX, TrainingSpec
from physrealx.errors import PhysRealXError

WORKSPACE = PROJECT_ROOT / ".physrealx/examples/05-data-training"
OUTPUT_DIR = PROJECT_ROOT / "examples/data_management_and_training/output"
px = PhysRealX.open(WORKSPACE)
receipts = {}

## 1. 转换训练数据：`datasets.ingest`

**用途**：将已验证 Episode 集合转换为统一数据契约，例如 LeRobot v3。

**检查**：多模态时间对齐、坐标/单位、动作语义、终止标记和来源关联。


In [ ]:
episode_collection = None  # 替换为 FINALIZED VerifiedEpisodeCollection/1 ArtifactRef
dataset_ingestion_operation = None
if episode_collection is None:
    print("[BLOCKED] 请在本单元格绑定 Episode 集合。")
else:
    try:
        dataset_ingestion_operation = px.datasets.ingest(
            episode_collection,
            method="lerobot_v3",
            config={"video_codec": "mp4"},
        )
        receipts["dataset_ingestion"] = operation_receipt(dataset_ingestion_operation)
        print(json.dumps(dataset_ingestion_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 2. 发布数据快照：`datasets.snapshot`

**用途**：固定数据文件、Episode 清单、划分规则和来源，作为训练的不可变输入。

**阻断条件**：资产/场景/任务跨 split 泄漏或时间错位时禁止发布。


In [ ]:
dataset_bundle = None  # 替换为已验证 DatasetBundle/1 ArtifactRef
snapshot_operation = None
if dataset_bundle is None:
    print("[BLOCKED] 请在本单元格绑定已验证数据包。")
else:
    try:
        snapshot_operation = px.datasets.snapshot(
            dataset_bundle,
            method="dvc",
            config={},
        )
        receipts["dataset_snapshot"] = operation_receipt(snapshot_operation)
        print(json.dumps(snapshot_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 3. 训练平台内置策略：`training.submit`

**用途**：提交具有明确数据版本、训练器、配置和种子的 `TrainingSpec`。

**输出**：平台注册的策略 `ModelManifest`；训练损失不代替固定基准评测。


In [ ]:
training_dataset = None  # 替换为已注册 Dataset VersionedRef
policy_model = None
if training_dataset is None:
    print("[BLOCKED] 请在本单元格绑定训练数据版本。")
else:
    try:
        trainer = px.methods.bind(
            MethodCapability.POLICY_TRAIN,
            "lerobot_act",
            config={"seed": 2026, "epochs": 100},
        )
        policy_model = px.training.submit(
            TrainingSpec(
                resource_id="cabinet2.act",
                version="1.0.0",
                dataset=training_dataset,
                trainer=trainer,
            )
        )
        receipts["policy_model"] = policy_model
        print(policy_model)
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 4. 执行隔离训练配方：`recipes.execute`

**用途**：在已安装的外部 Trainer 上分别运行策略或世界模型配方。

**契约**：两类训练产生不同模型包，不用一个模型类型混装能力。


In [ ]:
policy_training_recipe = None  # 替换为已构造的策略 RecipeExecutionSpec
world_model_training_recipe = None  # 替换为世界模型 RecipeExecutionSpec
policy_training_execution = None
world_model_training_execution = None
if policy_training_recipe is None:
    print("[BLOCKED] 未配置外部策略训练配方。")
else:
    try:
        policy_training_execution = px.recipes.execute(policy_training_recipe)
        receipts["policy_training_execution"] = policy_training_execution.execution
        print(policy_training_execution.execution)
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")
if world_model_training_recipe is None:
    print("[BLOCKED] 未配置世界模型训练配方。")
else:
    try:
        world_model_training_execution = px.recipes.execute(world_model_training_recipe)
        receipts["world_model_training_execution"] = world_model_training_execution.execution
        print(world_model_training_execution.execution)
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 5. 登记外部模型：`models.register`

**用途**：将外部 Trainer 的模型包与数据、代码、环境、配置、种子和指标绑定。

**输出**：可部署、可回溯的模型版本。


In [ ]:
external_model = None  # 替换为通过校验的 ModelBundle/1 ArtifactRef
model_registration_operation = None
if external_model is None:
    print("[BLOCKED] 请在本单元格绑定外部模型包。")
else:
    try:
        model_registration_operation = px.models.register(
            external_model,
            method="mlflow",
            config={},
        )
        receipts["model_registration"] = operation_receipt(model_registration_operation)
        print(json.dumps(model_registration_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 6. 世界模型推演：`world_models.rollout`

**用途**：给定上下文和候选动作，生成模型声明的特征、视频、状态或价值预测。

**边界**：视觉质量指标不能代替物理一致性和闭环成功率。


In [ ]:
rollout_model = None  # 替换为世界模型 VersionedRef
rollout_context = None  # 替换为上下文 ArtifactRef
rollout_actions = None  # 替换为候选动作 ArtifactRef
rollout_operation = None
if None in (rollout_model, rollout_context, rollout_actions):
    print("[BLOCKED] 请同时配置 rollout_model、rollout_context 和 rollout_actions。")
else:
    try:
        rollout_operation = px.world_models.rollout(
            rollout_model,
            rollout_context,
            rollout_actions,
            method="dino_wm",
            config={"horizon": 16},
        )
        receipts["world_model_rollout"] = operation_receipt(rollout_operation)
        print(json.dumps(rollout_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 7. 保存回执

回执固定本次实际执行的数据、训练和模型操作。


In [ ]:
report = write_summary(
    OUTPUT_DIR / "training-report.json",
    {"module": "数据管理与模型训练", "receipts": receipts},
)
print(json.dumps(report, ensure_ascii=False, indent=2, default=str))